# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [1]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    with open(filename, 'a') as file:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        file.write(f"{timestamp} - {text}\n") 

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    try:
        with open(filename, 'r') as file:
            for line in file:
                print(line.strip())
    except FileNotFoundError:
        print(f"Error: The file '{filename}' does not exist.")
        return
    except:
        print("Unexpected error")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    try:
        with open(filename, 'r') as file:
            lines = file.readlines()
            results = [line.strip() for line in lines if keyword.lower() in line.lower()]
            return results
    except FileNotFoundError:
        print(f"Error: The file '{filename}' does not exist.")
        return []
    except:
        print("Unexpected error")

def count_entries(filename):
    """Count and return the number of journal entries."""
    try:
        with open(filename, 'r') as file:
            lines = file.readlines()
            results = len([line for line in lines if line.strip()])
            return results
    except FileNotFoundError:
        print(f"Error: The file '{filename}' does not exist.")
        return 0
    except:
        print("Unexpected error")

def run_journal():
    """Menu-driven journal interface."""
    while True:
        print("\nJournal Menu:")
        print("[A]dd entry")
        print("[R]ead entries")
        print("[S]earch entries")
        print("[C]ount entries")
        print("[Q]uit")
        
        choice = input("Choose an option: ").strip().upper()
        
        if choice == 'A':
            text = input("Enter your journal entry: ")
            add_entry(JOURNAL_FILE, text)
            print("Entry added.")
        elif choice == 'R':
            read_journal(JOURNAL_FILE)
        elif choice == 'S':
            keyword = input("Enter a keyword to search for: ")
            results = search_journal(JOURNAL_FILE, keyword)
            if results:
                print("Search results:")
                for line in results:
                    print(line)
            else:
                print("No entries found with that keyword.")
        elif choice == 'C':
            total_entries = count_entries(JOURNAL_FILE)
            print(f"Total entries: {total_entries}")
        elif choice == 'Q':
            print("Exiting the journal. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")


# --- QUICK TEST (non-interactive) ---
add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
read_journal(JOURNAL_FILE)
print("Search results:", search_journal(JOURNAL_FILE, "python"))
print("Total entries:", count_entries(JOURNAL_FILE))

run_journal()  # Uncomment to test the interactive menu

2026-10-13 14:58:54 - Started learning about file I/O today!
2026-10-13 14:58:54 - Python makes reading files so easy.
2026-10-13 15:03:13 - Started learning about file I/O today!
2026-10-13 15:03:13 - Python makes reading files so easy.
2026-10-13 15:03:30 - No paras de tocar los cojones
2026-10-15 13:32:34 - Started learning about file I/O today!
2026-10-15 13:32:34 - Python makes reading files so easy.
Search results: ['2026-10-13 14:58:54 - Python makes reading files so easy.', '2026-10-13 15:03:13 - Python makes reading files so easy.', '2026-10-15 13:32:34 - Python makes reading files so easy.']
Total entries: 7

Journal Menu:
[A]dd entry
[R]ead entries
[S]earch entries
[C]ount entries
[Q]uit


Entry added.

Journal Menu:
[A]dd entry
[R]ead entries
[S]earch entries
[C]ount entries
[Q]uit


Exiting the journal. Goodbye!


---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [3]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from JSON. Return [] if missing or corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            tasks = json.load(file)
        if not isinstance(tasks, list):
            return []
        for task in tasks:
            if not isinstance(task, dict):
                return []
            if not isinstance(task.get("title"), str) or not task["title"].strip():
                return []
            if task.get("priority") not in ["high", "medium", "low"]:
                return []
            if not isinstance(task.get("done"), bool):
                return []
        return tasks
    except (FileNotFoundError, json.JSONDecodeError, UnicodeDecodeError):
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(tasks, file, indent=2, ensure_ascii=False)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    title = input("Task title: ").strip()
    while not title:
        title = input("Title cannot be empty. Task title: ").strip()

    priority = input("Priority (high/medium/low): ").strip().lower()
    while priority not in ["high", "medium", "low"]:
        priority = input("Choose high, medium, or low: ").strip().lower()

    tasks.append({"title": title, "priority": priority, "done": False})
    print("Task added.")

def list_tasks(tasks):
    """Display all tasks in a formatted table using zero-based indices."""
    if not tasks:
        print("No tasks yet.")
        return

    print(f"{'Index':<7} {'Status':<10} {'Title':<30} Priority")
    print("-" * 60)
    for index, task in enumerate(tasks):
        status = "Done" if task["done"] else "Pending"
        print(f"{index:<7} {status:<10} {task['title']:<30} {task['priority']}")

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    if not tasks:
        print("No tasks to mark as done.")
        return

    list_tasks(tasks)
    try:
        index = int(input("Task index to mark as done: "))
    except ValueError:
        print("Please enter a whole number.")
        return

    if 0 <= index < len(tasks):
        tasks[index]["done"] = True
        print("Task marked as done.")
    else:
        print("Invalid task index.")

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    if not tasks:
        print("No tasks to remove.")
        return

    list_tasks(tasks)
    try:
        index = int(input("Task index to remove: "))
    except ValueError:
        print("Please enter a whole number.")
        return

    if 0 <= index < len(tasks):
        removed_task = tasks.pop(index)
        print(f"Removed: {removed_task['title']}")
    else:
        print("Invalid task index.")

def run_todo():
    """Main menu loop with auto-save."""
    tasks = load_tasks(TODO_FILE)
    while True:
        print("\n[A]dd  [L]ist  [D]one  [R]emove  [Q]uit")
        choice = input("Choose an option: ").strip().upper()

        if choice == "A":
            add_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "L":
            list_tasks(tasks)
        elif choice == "D":
            mark_done(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "R":
            delete_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "Q":
            print("Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")


# --- QUICK TEST (non-interactive) ---
tasks = load_tasks(TODO_FILE)
tasks.append({"title": "Test task", "priority": "high", "done": False})
save_tasks(TODO_FILE, tasks)
tasks = load_tasks(TODO_FILE)
list_tasks(tasks)

# run_todo()  # Uncomment to test the interactive menu

Index   Status     Title                          Priority
------------------------------------------------------------
0       Pending    Test task                      high
1       Pending    Test task                      high
2       Pending    Test task                      high


---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [1]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [ ]:
import csv
import math

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    expenses = []
    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            for row in reader:
                category = row.get("category")
                if not category or not category.strip():
                    continue
                try:
                    amount = float(row.get("amount"))
                except (ValueError, TypeError):
                    continue
                if not math.isfinite(amount):
                    continue

                row["category"] = category.strip()
                row["amount"] = amount
                expenses.append(row)
    except FileNotFoundError:
        print(f"File not found: {filename}")
    return expenses

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        if category not in totals:
            totals[category] = 0.0
        totals[category] += expense["amount"]
    return totals

def most_expensive(expenses):
    """Return the most expensive entry, or None if there are no expenses."""
    if not expenses:
        return None

    highest = expenses[0]
    for expense in expenses:
        if expense["amount"] > highest["amount"]:
            highest = expense
    return highest

def print_report(expenses):
    """Print total spending, category totals, and the most expensive item."""
    if not expenses:
        print("No valid expenses to report.")
        return

    total = 0.0
    for expense in expenses:
        total += expense["amount"]

    print("===== EXPENSE REPORT =====")
    print(f"Total spent: {total:.2f}")
    print("\nSpending by category:")
    totals = total_by_category(expenses)
    for category, amount in totals.items():
        print(f"  {category}: {amount:.2f}")

    highest = most_expensive(expenses)
    print("\nMost expensive item:")
    print(f"  {highest['description']} ({highest['category']})")
    print(f"  Date: {highest['date']}")
    print(f"  Amount: {highest['amount']:.2f}")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)

---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [ ]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

In [ ]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r", encoding="utf-8") as file:
            records = json.load(file)
    except (OSError, json.JSONDecodeError, UnicodeDecodeError) as error:
        report["errors"].append(f"Could not read JSON: {error}")
        return report

    if not isinstance(records, list):
        report["errors"].append("JSON root must be a list.")
        return report

    for index, record in enumerate(records, start=1):
        if not isinstance(record, dict):
            report["invalid"] += 1
            report["errors"].append(f"Record {index}: expected a dictionary.")
            continue
        missing_keys = [key for key in required_keys if key not in record]
        if missing_keys:
            report["invalid"] += 1
            report["errors"].append(f"Record {index}: missing keys {missing_keys}.")
        else:
            report["valid"] += 1
    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.reader(file, strict=True)
            header = next(reader, None)
            if header is None:
                report["errors"].append("CSV file is empty.")
                return report
            if len(header) != expected_columns:
                report["errors"].append(
                    f"Header: expected {expected_columns} columns, got {len(header)}."
                )
            for row_number, row in enumerate(reader, start=2):
                if len(row) == expected_columns:
                    report["valid"] += 1
                else:
                    report["invalid"] += 1
                    report["errors"].append(
                        f"Row {row_number}: expected {expected_columns} fields, got {len(row)}."
                    )
    except (OSError, csv.Error, UnicodeDecodeError) as error:
        report["errors"].append(f"Could not read CSV: {error}")
    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    print(f"Validation report: {filename}")
    print(f"Valid records: {report['valid']}")
    print(f"Invalid records: {report['invalid']}")
    if report["errors"]:
        print("Errors:")
        for error in report["errors"]:
            print(f"  - {error}")
    else:
        print("No errors found.")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [ ]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

In [ ]:
import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        with open(json_filename, "r", encoding="utf-8") as file:
            records = json.load(file)
        if not isinstance(records, list) or any(not isinstance(record, dict) for record in records):
            print("Expected a JSON list of dictionaries.")
            return
        if not records:
            print("No records to export.")
            return
        fieldnames = list(records[0])
        if not fieldnames or any(set(record) != set(fieldnames) for record in records):
            print("Records must have the same non-empty set of keys.")
            return
        with open(csv_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(records)
        print(f"Exported {len(records)} record(s) to {csv_filename}.")
    except (OSError, json.JSONDecodeError, UnicodeDecodeError, csv.Error) as error:
        print(f"Export failed: {error}")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file, strict=True)
            headers = reader.fieldnames
            if not headers or any(not header.strip() for header in headers):
                print("CSV must have non-empty column headers.")
                return
            if len(set(headers)) != len(headers):
                print("CSV column headers must be unique.")
                return
            records = []
            for row_number, row in enumerate(reader, start=2):
                if None in row or None in row.values():
                    print(f"Row {row_number}: field count does not match the header.")
                    return
                records.append(row)
        with open(json_filename, "w", encoding="utf-8") as file:
            json.dump(records, file, indent=2, ensure_ascii=False)
        print(f"Exported {len(records)} record(s) to {json_filename}.")
    except (OSError, csv.Error, UnicodeDecodeError) as error:
        print(f"Conversion failed: {error}")

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    try:
        with open(json_filename, "r", encoding="utf-8") as file:
            records = json.load(file)
        if not isinstance(records, list) or any(not isinstance(record, dict) for record in records):
            print("Expected a JSON list of dictionaries.")
            return
        if not records:
            print("No records to export.")
            return
        fieldnames = list(records[0])
        if not fieldnames or any(set(record) != set(fieldnames) for record in records):
            print("Records must have the same non-empty set of keys.")
            return
        if filter_key not in fieldnames:
            print(f"Unknown filter key: {filter_key}")
            return
        records = [record for record in records if record[filter_key] == filter_value]
        with open(csv_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(records)
        print(f"Exported {len(records)} record(s) to {csv_filename}.")
    except (OSError, json.JSONDecodeError, UnicodeDecodeError, csv.Error) as error:
        print(f"Export failed: {error}")


# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")

---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [ ]:
import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            students = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError, UnicodeDecodeError):
        return []
    if not isinstance(students, list):
        return []
    for student in students:
        if not isinstance(student, dict):
            return []
        fields = [student.get(key) for key in ("name", "age", "email", "grade")]
        is_valid, error = validate_student(*fields)
        if not is_valid:
            return []
        student["age"] = int(student["age"])
    return students

def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(students, file, indent=2, ensure_ascii=False)

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    if not isinstance(name, str) or not name.strip():
        return False, "Name must not be empty."
    if isinstance(age, bool) or not isinstance(age, (str, int)):
        return False, "Age must be a positive integer."
    try:
        age = int(age)
    except ValueError:
        return False, "Age must be a positive integer."
    if age <= 0:
        return False, "Age must be a positive integer."
    if not isinstance(email, str) or "@" not in email:
        return False, "Email must contain '@'."
    if grade not in ("A", "B", "C", "D", "F"):
        return False, "Grade must be A, B, C, D, or F."
    return True, ""

# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    name = input("Name: ").strip()
    age = input("Age: ").strip()
    email = input("Email: ").strip()
    grade = input("Grade (A/B/C/D/F): ").strip().upper()
    is_valid, error = validate_student(name, age, email, grade)
    if not is_valid:
        print(error)
        return
    students.append({"name": name, "age": int(age), "email": email, "grade": grade})
    print("Student added.")

def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("No students found.")
        return
    print(f"{'Name':<25} {'Age':<5} {'Email':<35} Grade")
    print("-" * 75)
    for student in students:
        print(f"{student['name']:<25} {student['age']:<5} {student['email']:<35} {student['grade']}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    term = input("Name to search for: ").strip().casefold()
    matches = [student for student in students if term in student["name"].casefold()]
    list_students(matches)

def update_student(students):
    """Update a student's grade or email by name."""
    name = input("Name of student to update: ").strip().casefold()
    for student in students:
        if student["name"].casefold() != name:
            continue
        field = input("Field to update (grade/email): ").strip().lower()
        if field not in ("grade", "email"):
            print("Choose grade or email.")
            return
        value = input(f"New {field}: ").strip()
        if field == "grade":
            value = value.upper()
        updated = student.copy()
        updated[field] = value
        is_valid, error = validate_student(
            updated["name"], updated["age"], updated["email"], updated["grade"]
        )
        if not is_valid:
            print(error)
            return
        student[field] = value
        print("Student updated.")
        return
    print("Student not found.")

def delete_student(students):
    """Remove a student by name."""
    name = input("Name of student to delete: ").strip().casefold()
    for index, student in enumerate(students):
        if student["name"].casefold() == name:
            confirmation = input(f"Delete {student['name']}? (y/n): ").strip().lower()
            if confirmation == "y":
                students.pop(index)
                print("Student deleted.")
            else:
                print("Deletion cancelled.")
            return
    print("Student not found.")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    imported = 0
    skipped = 0
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file, strict=True)
            required = {"name", "age", "email", "grade"}
            if not reader.fieldnames or not required.issubset(reader.fieldnames):
                print("CSV must contain name, age, email, and grade columns.")
                return
            for row_number, row in enumerate(reader, start=2):
                if None in row or None in row.values():
                    skipped += 1
                    print(f"Skipped row {row_number}: field count does not match the header.")
                    continue
                name = row["name"].strip()
                age = row["age"].strip()
                email = row["email"].strip()
                grade = row["grade"].strip().upper()
                is_valid, error = validate_student(name, age, email, grade)
                if not is_valid:
                    skipped += 1
                    print(f"Skipped row {row_number}: {error}")
                    continue
                students.append({"name": name, "age": int(age), "email": email, "grade": grade})
                imported += 1
    except (OSError, csv.Error, UnicodeDecodeError) as error:
        print(f"Import failed: {error}")
    print(f"Imported {imported} student(s); skipped {skipped} row(s).")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    try:
        with open(csv_filename, "w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=["name", "age", "email", "grade"])
            writer.writeheader()
            writer.writerows(students)
        print(f"Exported {len(students)} student(s) to {csv_filename}.")
    except (OSError, csv.Error, UnicodeDecodeError) as error:
        print(f"Export failed: {error}")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

# run_student_db()  # Uncomment to test

In [ ]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")